# 🧹 Phase 1: Data Cleaning & Pre-processing

Based on the initial analysis, this notebook implements a **conservative cleaning strategy**.
We aim to fix structural errors (missing IDs) and optimize data types while preserving as much information as possible (e.g., keeping anime with partial data).

**Key Actions:**
1.  **Impute** missing values where logic allows (e.g., `favorites=0`, `is_rewatching=False`).
2.  **Drop** only structurally corrupted rows (missing Primary/Foreign Keys).
3.  **Extract** useful features (e.g., `year` from `start_date`).
4.  **Optimize** data types (float64 -> float32/16, objects -> categories) to reduce memory usage.
5.  **Save** to optimized Parquet format for the next steps.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import gc
from pathlib import Path

# 🧪 Standard Import (requires 'solution' package installed)
from solution.src.common import RAW_DATA_DIR, PROCESSED_DATA_DIR

# 📦 Import atomic cleaning functions and utils
from nb_01_cleaning import (
    REMAINING_FILES,
    optimize_dtypes,
    save_refined,
    print_fancy_progress,
    get_memory_usage_mb,
    clean_characters_logic,
    clean_details_logic,
    clean_stats_logic,
    clean_ratings_chunk_logic,
    clean_generic_file_logic
)

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print(f"Reading from: {RAW_DATA_DIR}")
print(f"Saving to:    {PROCESSED_DATA_DIR}")

## 1. Characters (`characters.csv`)
- Drop rows with missing `character_mal_id`.
- Fill missing `favorites` with 0.
- Drop `about` column (mostly empty/text heavy text not needed for general stats).

In [ ]:
# 1. Load
df_chars = pd.read_csv(RAW_DATA_DIR / "characters.csv")

# 2. Clean (Atomic Logic)
df_chars = clean_characters_logic(df_chars)

# 3. Optimize & Save
df_chars = optimize_dtypes(df_chars)
save_refined(df_chars, "characters.csv")

## 2. Anime Details (`details.csv`)
- **Year**: Derive from `start_date` where missing.
- **Dates**: Convert to datetime objects.
- **Missing Stats**: Keep `score`, `rank` as NaN (preserve "unranked" status).
- **Columns**: Keep `demographics` and others even if they look empty (might have sparse data).

In [ ]:
# 1. Load
df_details = pd.read_csv(RAW_DATA_DIR / "details.csv")

# 2. Clean (Atomic Logic)
df_details = clean_details_logic(df_details)

# 3. Optimize & Save
df_details = optimize_dtypes(df_details)
save_refined(df_details, "details.csv")

## 3. Anime Stats (`stats.csv`)
- **Votes**: Fill NaN score votes with 0.
- **Percentages**: Fill NaN percentages with 0.
- **Conserve Rows**: Do not drop rows just because they have no votes.

In [ ]:
# 1. Load
df_stats = pd.read_csv(RAW_DATA_DIR / "stats.csv")

# 2. Clean (Atomic Logic)
df_stats = clean_stats_logic(df_stats)

# 3. Optimize & Save
df_stats = optimize_dtypes(df_stats)
save_refined(df_stats, "stats.csv")

## 4. User Ratings (`ratings.csv`)
- **Huge File**: Process carefully.
- **Username**: Drop NaN (orphan votes).
- **Rewatching**: Assume `False` (0) if missing.

In [ ]:
# Configuration
chunk_size = 5_000_000
chunks = []
csv_path = RAW_DATA_DIR / "ratings.csv"

# Global Metrics accumulating
total_rows_read = 0
total_dropped_username = 0
total_rewatch_imputed = 0

print(f"Processing {csv_path} in chunks...")
ESTIMATED_CHUNKS = 25 

with pd.read_csv(csv_path, chunksize=chunk_size) as reader:
    for i, chunk in enumerate(reader, 1):
        print_fancy_progress(i, ESTIMATED_CHUNKS, prefix="Chunks:")
        
        # --- ATOMIC CLEANING ---
        chunk, metrics = clean_ratings_chunk_logic(chunk)
        
        # Accumulate metrics
        total_rows_read += metrics['rows_read']
        total_dropped_username += metrics['dropped_username']
        total_rewatch_imputed += metrics['rewatch_imputed']
        
        chunks.append(chunk)

print() # Newline

# Concatenate & Save
print("Concatenation...")
df_ratings = pd.concat(chunks, ignore_index=True)
print("Concatenation complete.")

# Printing Report Manually here since logic is chunked
print(f"--- Ratings Cleaning Report ---")
print(f"Total Rows Read: {total_rows_read}")
print(f"Rows Dropped (Missing Username): {total_dropped_username}")
print(f"Rewatching Flags Imputed: {total_rewatch_imputed}")
print(f"Final Shape: {df_ratings.shape}")

df_ratings = optimize_dtypes(df_ratings)
save_refined(df_ratings, "ratings.csv")

del df_ratings, chunks
_ = gc.collect()

## 5. Other Files
Processing the remaining files with standard optimizations:
- `person_details.csv`
- `profiles.csv`
- `favs.csv`
- `recommendations.csv`
- Relational tables (`*_works.csv`, `*_nicknames.csv`)

In [ ]:
total_files = len(REMAINING_FILES)
results = []
print_fancy_progress(0, total_files, prefix="Files :")

for i, filename in enumerate(REMAINING_FILES, 1):
    file_path = RAW_DATA_DIR / filename
    if os.path.exists(file_path):
        # 1. Read
        df = pd.read_csv(file_path)
        
        # Stats Capture
        initial_rows = len(df)
        initial_mem = get_memory_usage_mb(df)
        
        # 2. Clean (Atomic)
        df, actions = clean_generic_file_logic(df, filename)

        # 3. Optimize
        df = optimize_dtypes(df, verbose=False)
        
        # Capture Final Stats
        final_rows = len(df)
        final_mem = get_memory_usage_mb(df)
        
        # 4. Save
        save_refined(df, filename, verbose=False)
        
        results.append({
            'file': filename, 
            'rows_src': initial_rows, 
            'rows_dst': final_rows,
            'mem_src': initial_mem,
            'mem_dst': final_mem,
            'actions': actions
        })
        
        print_fancy_progress(i, total_files, prefix="Files :")

# Print Summary
print(f"\n--- Other Files Cleaning Report ---")
for res in results:
    mem_red = (res['mem_src'] - res['mem_dst']) / res['mem_src'] * 100 if res['mem_src'] > 0 else 0
    print(f"[{res['file']}]")
    print(f"  Rows:   {res['rows_src']} -> {res['rows_dst']}")
    print(f"  Memory: {res['mem_src']:.2f} MB -> {res['mem_dst']:.2f} MB ({mem_red:.1f}% reduction)")
    if res['actions']:
        print(f"  Actions: {', '.join(res['actions'])}")
    print("-" * 30)
print(f"Total files processed: {len(results)}")